# vector Database
- FAISS
- Chroma
- pinecone
- qdrant
- Astra

# FAISS

In [1]:

!uv pip install -q langchain langchain_community pypdf faiss-cpu langchain_huggingface langchain-text-splitters

In [2]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
import faiss
from langchain_community.docstore.in_memory import InMemoryDocstore

/tmp/ipykernel_15245/1239437360.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


In [3]:
sample_documents = [
    Document(
        page_content="""
        Artificial Intelligence (AI) is the simulation of human intelligence in machines.
        These systems are designed to think like humans and mimic their actions.
        AI can be categorized into narrow AI and general AI.
        """,
        metadata={"source": "AI Introduction", "page": 1, "topic": "AI"}
    ),

    Document(
        page_content="""
        Machine Learning is a subset of AI that enables systems to learn from data.
        Instead of being explicitly programmed, ML algorithms find patterns in data.
        Common types include supervised, unsupervised, and reinforcement learning.
        """,
        metadata={"source": "ML Basics", "page": 1, "topic": "ML"}
    ),

    Document(
        page_content="""
        Deep Learning is a subset of machine learning based on artificial neural networks.
        It uses multiple layers to progressively extract higher-level features from raw input.
        Deep learning has revolutionized computer vision, NLP, and speech recognition.
        """,
        metadata={"source": "Deep Learning", "page": 1, "topic": "DL"}
    ),

    Document(
        page_content="""
        Natural Language Processing (NLP) is a branch of AI that helps computers understand human language.
        It combines computational linguistics with machine learning and deep learning models.
        Applications include chatbots, translation, sentiment analysis, and text summarization.
        """,
        metadata={"source": "NLP Overview", "page": 2, "topic": "NLP"}
    ),

    Document(
        page_content="""
        Supervised learning is a type of machine learning where models learn from labeled data.
        Each training example contains an input and its corresponding correct output.
        Common supervised learning tasks include classification and regression.
        Examples include spam detection, house price prediction, and image classification.
        """,
        metadata={"source": "Supervised Learning", "page": 2, "topic": "ML"}
    ),

    Document(
        page_content="""
        Unsupervised learning allows machine learning models to discover patterns in data without labeled outputs.
        Clustering and dimensionality reduction are common unsupervised learning techniques.
        K-means clustering is frequently used to group similar data points.
        Unsupervised learning can be useful for customer segmentation and anomaly detection.
        """,
        metadata={"source": "Unsupervised Learning", "page": 2, "topic": "ML"}
    ),

    Document(
        page_content="""
        Reinforcement learning is a machine learning approach where an agent learns by interacting with an environment.
        The agent takes actions and receives rewards or penalties based on its behavior.
        Over time, the agent learns a policy that helps maximize its total reward.
        Reinforcement learning is used in robotics, games, and autonomous systems.
        """,
        metadata={"source": "Reinforcement Learning", "page": 3, "topic": "ML"}
    ),

    Document(
        page_content="""
        Neural networks are machine learning models inspired by the structure of biological brains.
        They consist of interconnected nodes called neurons organized into layers.
        A typical neural network contains an input layer, hidden layers, and an output layer.
        Neural networks can learn complex relationships by adjusting their weights during training.
        """,
        metadata={"source": "Neural Networks", "page": 3, "topic": "DL"}
    ),

    Document(
        page_content="""
        Computer Vision is a field of AI that enables computers to understand and analyze images and videos.
        Deep learning models such as convolutional neural networks are commonly used for visual tasks.
        Applications include object detection, facial recognition, medical image analysis, and image classification.
        Computer vision systems learn visual patterns from large collections of images.
        """,
        metadata={"source": "Computer Vision", "page": 3, "topic": "Computer Vision"}
    ),

    Document(
        page_content="""
        Generative AI is a branch of artificial intelligence that can create new content based on learned patterns.
        Generative models can produce text, images, audio, video, and computer code.
        Large language models are a major example of generative AI used for generating and understanding text.
        Popular applications include AI assistants, content generation, code generation, and document summarization.
        """,
        metadata={"source": "Generative AI", "page": 4, "topic": "GenAI"}
    )
]

print(sample_documents)

[Document(metadata={'source': 'AI Introduction', 'page': 1, 'topic': 'AI'}, page_content='\n        Artificial Intelligence (AI) is the simulation of human intelligence in machines.\n        These systems are designed to think like humans and mimic their actions.\n        AI can be categorized into narrow AI and general AI.\n        '), Document(metadata={'source': 'ML Basics', 'page': 1, 'topic': 'ML'}, page_content='\n        Machine Learning is a subset of AI that enables systems to learn from data.\n        Instead of being explicitly programmed, ML algorithms find patterns in data.\n        Common types include supervised, unsupervised, and reinforcement learning.\n        '), Document(metadata={'source': 'Deep Learning', 'page': 1, 'topic': 'DL'}, page_content='\n        Deep Learning is a subset of machine learning based on artificial neural networks.\n        It uses multiple layers to progressively extract higher-level features from raw input.\n        Deep learning has revolu

In [4]:
# text splitting
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    length_function=len,
    separators=[" "]
)

# Split the documents into chunks
chunks = text_splitter.split_documents(sample_documents)
print(chunks[0])
print(chunks[1])

page_content='Artificial Intelligence (AI) is the simulation of human intelligence in machines.
        These systems are designed to think like humans and mimic their actions.
        AI can be categorized into narrow AI and general AI.' metadata={'source': 'AI Introduction', 'page': 1, 'topic': 'AI'}
page_content='Machine Learning is a subset of AI that enables systems to learn from data.
        Instead of being explicitly programmed, ML algorithms find patterns in data.
        Common types include supervised, unsupervised, and reinforcement learning.' metadata={'source': 'ML Basics', 'page': 1, 'topic': 'ML'}


In [5]:
embeddings = HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [6]:
embeddings_size =len(embeddings.embed_query("Hello"))
print(f"Word embedding size: {embeddings_size}")

Word embedding size: 384


| Family | Main idea                 |
| ------ | ------------------------- |
| Flat   | Compare with every vector |
| IVF    | Search selected clusters  |
| HNSW   | Navigate through a graph  |


In [7]:

# IndexFlatL2 = brute-force search comparing every vector (simplest FAISS index)
index = faiss.IndexFlatL2(embeddings_size)

In [8]:
# use HNSW with inner product

# import faiss
# import numpy as np

# dimension = 768
# M = 32  # Har vector ke approximate graph connections

# faiss_index = faiss.IndexHNSWFlat(
#     dimension,
#     M,
#     faiss.METRIC_INNER_PRODUCT
# )

# # Higher value = better recall, but slower indexing
# faiss_index.hnsw.efConstruction = 200

# # Higher value = better recall, but slower search
# faiss_index.hnsw.efSearch = 64


# -------------------------------------------------------------------
# use euclident distance

# faiss_index = faiss.IndexHNSWFlat(
#     dimension,
#     M,
#     faiss.METRIC_L2
# )

# faiss_index.hnsw.efConstruction = 200
# faiss_index.hnsw.efSearch = 64

In [9]:

vector_store = FAISS(
    embedding_function=embeddings,
    index=index,
    docstore=InMemoryDocstore(),
    index_to_docstore_id={},
)

vector_store.add_documents(chunks)

['e4013bdb-7911-4f83-90b3-1001fe4a8089',
 '8b8bb7f9-e7f3-4cb6-bb00-892e9484b621',
 '7d6c02c1-f005-4a08-b0d2-83b96336bd5e',
 'e611d035-ff80-4010-b2a9-fcc42148fc97',
 '9476d259-e7d6-47da-a65f-2a6112c46b07',
 '3cb2721e-5d08-4314-ab6d-996c6591dab3',
 '7d371d01-57ea-419e-8759-0549f09ad9a8',
 'd6e4a238-7957-44d5-b659-dded2a551e60',
 '1362ebb5-b117-4e8f-a666-656c9d764d50',
 '17d1db8e-2a82-4b66-8efb-14592766239c']

In [10]:
## Save vector tore for later use
vector_store.save_local("faiss_index")
print("Vector store saved to 'faiss_index' directory")

Vector store saved to 'faiss_index' directory


In [11]:
# Load vector store
loaded_vectorstore = FAISS.load_local(
    "faiss_index",
    embeddings,
    allow_dangerous_deserialization=True
)

print(f"Loaded vector store contains {loaded_vectorstore.index.ntotal} vectors")

Loaded vector store contains 10 vectors


In [12]:
# Similarity Search
query = "What is deep learning"

results = vector_store.similarity_search(query, k=3)
print(results)

[Document(id='7d6c02c1-f005-4a08-b0d2-83b96336bd5e', metadata={'source': 'Deep Learning', 'page': 1, 'topic': 'DL'}, page_content='Deep Learning is a subset of machine learning based on artificial neural networks.\n        It uses multiple layers to progressively extract higher-level features from raw input.\n        Deep learning has revolutionized computer vision, NLP, and speech recognition.'), Document(id='1362ebb5-b117-4e8f-a666-656c9d764d50', metadata={'source': 'Computer Vision', 'page': 3, 'topic': 'Computer Vision'}, page_content='Computer Vision is a field of AI that enables computers to understand and analyze images and videos.\n        Deep learning models such as convolutional neural networks are commonly used for visual tasks.\n        Applications include object detection, facial recognition, medical image analysis, and image classification.\n        Computer vision systems learn visual patterns from large collections of images.'), Document(id='d6e4a238-7957-44d5-b659-dd

In [13]:
print(f"Query: {query}\n")
print("Top 3 similar chunks:")

for i, doc in enumerate(results):
    print(f"\n{i+1}. Source: {doc.metadata['source']}")
    print(f"   Content: {doc.page_content[:200]}...")

Query: What is deep learning

Top 3 similar chunks:

1. Source: Deep Learning
   Content: Deep Learning is a subset of machine learning based on artificial neural networks.
        It uses multiple layers to progressively extract higher-level features from raw input.
        Deep learning ...

2. Source: Computer Vision
   Content: Computer Vision is a field of AI that enables computers to understand and analyze images and videos.
        Deep learning models such as convolutional neural networks are commonly used for visual tas...

3. Source: Neural Networks
   Content: Neural networks are machine learning models inspired by the structure of biological brains.
        They consist of interconnected nodes called neurons organized into layers.
        A typical neural ...


In [14]:
# Similarity Search with score
results_with_scores = vector_store.similarity_search_with_score(query, k=3)

print("\n\nSimilarity search with scores:")
for doc, score in results_with_scores:
    print(f"\nScore: {score:.3f}")
    print(f"Source: {doc.metadata['source']}")
    print(f"Content preview: {doc.page_content[:100]}...")



Similarity search with scores:

Score: 0.343
Source: Deep Learning
Content preview: Deep Learning is a subset of machine learning based on artificial neural networks.
        It uses m...

Score: 0.725
Source: Computer Vision
Content preview: Computer Vision is a field of AI that enables computers to understand and analyze images and videos....

Score: 1.009
Source: Neural Networks
Content preview: Neural networks are machine learning models inspired by the structure of biological brains.
        ...


In [15]:
# Search with metadata filtering
filter_dict = {"topic":"ML"}
filtered_results =vector_store.similarity_search(
    query,
    k=3,
    filter=filter_dict
)
print(filtered_results)

[Document(id='9476d259-e7d6-47da-a65f-2a6112c46b07', metadata={'source': 'Supervised Learning', 'page': 2, 'topic': 'ML'}, page_content='Supervised learning is a type of machine learning where models learn from labeled data.\n        Each training example contains an input and its corresponding correct output.\n        Common supervised learning tasks include classification and regression.\n        Examples include spam detection, house price prediction, and image classification.'), Document(id='8b8bb7f9-e7f3-4cb6-bb00-892e9484b621', metadata={'source': 'ML Basics', 'page': 1, 'topic': 'ML'}, page_content='Machine Learning is a subset of AI that enables systems to learn from data.\n        Instead of being explicitly programmed, ML algorithms find patterns in data.\n        Common types include supervised, unsupervised, and reinforcement learning.'), Document(id='7d371d01-57ea-419e-8759-0549f09ad9a8', metadata={'source': 'Reinforcement Learning', 'page': 3, 'topic': 'ML'}, page_content

# Chroma

| Feature                | FAISS   | ChromaDB |
| ---------------------- | ------- | -------- |
| Flat index             | ✅       | ❌        |
| IVF                    | ✅       | ❌        |
| HNSW                   | ✅       | ✅        |
| PQ                     | ✅       | ❌        |
| Manual index selection | ✅       | ❌        |
| Metadata filtering     | Limited | ✅        |
| Persistence            | Manual  | Built-in |
| LangChain integration  | ✅       | ✅        |


Final understanding
FAISS = vector index/search engine

You manually manage:
- index type
- dimension
- metric
- docstore
- ID mapping
- persistence
  
Chroma = vector database
It manages:
- vectors
- documents
- metadata
- IDs
- collections
- index
- persistence


| Feature         | FAISS                 | Chroma |
| --------------- | --------------------- | ------ |
| Store vectors   | ✅                     | ✅      |
| Store documents | ❌                     | ✅      |
| Store metadata  | ❌                     | ✅      |
| Collections     | ❌                     | ✅      |
| CRUD            | Limited               | ✅      |
| Filtering       | ❌ (native)            | ✅      |
| Persistence     | Basic index save/load | ✅      |
| Client APIs     | ❌                     | ✅      |
| Server mode     | ❌                     | ✅      |


| Component        | LangChain + FAISS             | Chroma                          |
| ---------------- | ----------------------------- | ------------------------------- |
| Embeddings       | Native FAISS index            | Chroma vector index             |
| Documents        | LangChain `Docstore`          | Chroma collection               |
| Metadata         | LangChain `Document.metadata` | Chroma collection record        |
| Mapping          | `index_to_docstore_id`        | Internally managed              |
| Save             | FAISS file + pickle           | Database persistence            |
| Metadata filters | Wrapper/application handling  | Native database filtering       |
| Collections      | Not native to FAISS           | Native                          |
| CRUD             | Wrapper/index-dependent       | Native record operations        |
| Server/cloud     | Separate system required      | Supported database architecture |


In [16]:
!uv pip install -q langchain_chroma

In [17]:
from langchain_chroma import Chroma
# Vectorstores
from langchain_community.vectorstores import Chroma

In [18]:
# --------------------------------------------------
# 5. Create Chroma vector store
# --------------------------------------------------
vector_store = Chroma(
    collection_name="first_db_chroma",
    embedding_function=embeddings,
    persist_directory="./chroma_db",
    collection_metadata={
        "hnsw:space": "cosine"
    }
)

/tmp/ipykernel_15245/2684770618.py:4: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vector_store = Chroma(


In [19]:
# --------------------------------------------------
# 6. Add documents
# --------------------------------------------------

document_ids = vector_store.add_documents(
    documents=chunks
)

print("Documents added:", len(document_ids))

print(
    "Total documents stored:",
    vector_store._collection.count()
)

Documents added: 10
Total documents stored: 14


In [20]:
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
)

In [21]:
retriever

VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x781bf786deb0>, search_kwargs={'k': 3})

In [22]:
retriever_01 = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={
        "k": 2,
        "filter": {
            "page": 2
        }
    }
)

In [23]:
retriever_01

VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x781bf786deb0>, search_kwargs={'k': 2, 'filter': {'page': 1}})

In [24]:
results = vector_store.similarity_search(
    query="What is reinforcement learning?",
    k=5,
    filter={
        "page": 3
    }
)

In [25]:
results

[Document(metadata={'topic': 'ML', 'source': 'ML Basics', 'page': 1}, page_content='Machine Learning is a subset of AI that enables systems to learn from data.\n        Instead of being explicitly programmed, ML algorithms find patterns in data.\n        Common types include supervised, unsupervised, and reinforcement learning.'),
 Document(metadata={'topic': 'ML', 'page': 1, 'source': 'ML Basics'}, page_content='Machine Learning is a subset of AI that enables systems to learn from data.\n        Instead of being explicitly programmed, ML algorithms find patterns in data.\n        Common types include supervised, unsupervised, and reinforcement learning.'),
 Document(metadata={'topic': 'AI', 'source': 'AI Introduction', 'page': 1}, page_content='Artificial Intelligence (AI) is the simulation of human intelligence in machines.\n        These systems are designed to think like humans and mimic their actions.\n        AI can be categorized into narrow AI and general AI.'),
 Document(metad

# Method 2 for Chroma

In [26]:
# Create a chromadb vector store
persist_directory = "./chroma_db"

# Initialize Chromadb with Open AI embeddings
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=persist_directory,
    collection_name="rag_collection"
)

print(f"Vector store created with {vectorstore._collection.count()} vectors")
print(f"Persisted to: {persist_directory}")

Vector store created with 14 vectors
Persisted to: ./chroma_db


In [27]:
query = "What are the types of machine learning?"

similar_docs=vectorstore.similarity_search(query, k=3)
similar_docs

[Document(metadata={'page': 1, 'topic': 'ML', 'source': 'ML Basics'}, page_content='Machine Learning is a subset of AI that enables systems to learn from data.\n        Instead of being explicitly programmed, ML algorithms find patterns in data.\n        Common types include supervised, unsupervised, and reinforcement learning.'),
 Document(metadata={'topic': 'ML', 'page': 1, 'source': 'ML Basics'}, page_content='Machine Learning is a subset of AI that enables systems to learn from data.\n        Instead of being explicitly programmed, ML algorithms find patterns in data.\n        Common types include supervised, unsupervised, and reinforcement learning.'),
 Document(metadata={'source': 'Supervised Learning', 'topic': 'ML', 'page': 2}, page_content='Supervised learning is a type of machine learning where models learn from labeled data.\n        Each training example contains an input and its corresponding correct output.\n        Common supervised learning tasks include classification 

In [28]:
print(f"Query: {query}")
print(f"\nTop {len(similar_docs)} similar chunks:")

for i, doc in enumerate(similar_docs):
    print(f"\n--- Chunk {i+1} ---")
    print(doc.page_content[:200] + "...")
    print(f"Source: {doc.metadata.get('source', 'Unknown')}")

Query: What are the types of machine learning?

Top 3 similar chunks:

--- Chunk 1 ---
Machine Learning is a subset of AI that enables systems to learn from data.
        Instead of being explicitly programmed, ML algorithms find patterns in data.
        Common types include supervised...
Source: ML Basics

--- Chunk 2 ---
Machine Learning is a subset of AI that enables systems to learn from data.
        Instead of being explicitly programmed, ML algorithms find patterns in data.
        Common types include supervised...
Source: ML Basics

--- Chunk 3 ---
Supervised learning is a type of machine learning where models learn from labeled data.
        Each training example contains an input and its corresponding correct output.
        Common supervised ...
Source: Supervised Learning


In [29]:
results_scores = vectorstore.similarity_search_with_score(query, k=3)
results_scores

[(Document(metadata={'page': 1, 'source': 'ML Basics', 'topic': 'ML'}, page_content='Machine Learning is a subset of AI that enables systems to learn from data.\n        Instead of being explicitly programmed, ML algorithms find patterns in data.\n        Common types include supervised, unsupervised, and reinforcement learning.'),
  0.48084166646003723),
 (Document(metadata={'topic': 'ML', 'source': 'ML Basics', 'page': 1}, page_content='Machine Learning is a subset of AI that enables systems to learn from data.\n        Instead of being explicitly programmed, ML algorithms find patterns in data.\n        Common types include supervised, unsupervised, and reinforcement learning.'),
  0.48084166646003723),
 (Document(metadata={'page': 2, 'topic': 'ML', 'source': 'Supervised Learning'}, page_content='Supervised learning is a type of machine learning where models learn from labeled data.\n        Each training example contains an input and its corresponding correct output.\n        Commo

# pinecone

In [34]:
!uv pip install -q langchain-pinecone  pinecone

In [31]:
from google.colab import userdata
api_key = userdata.get('pinecone')

In [32]:
api_key

'pcsk_2fyqA4_J8T7AGnKrZF3qfdHsx3AfJVb8fMHSLonBkXzuFLuLGbaZZ1atndF3y19EfpgVqS'

In [35]:
from pinecone import Pinecone
pc = Pinecone(api_key=api_key)

In [39]:
# pc = Pinecone(api_key=os.getenv("PINECONE_API_KEY"))
import time
from pinecone import ServerlessSpec

index_name = "langchain-llama-index"

if not pc.has_index(index_name):
    pc.create_index(
        name=index_name,
        dimension=384,
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1"),
    )

# Wait until the index is ready before using it
while not pc.describe_index(index_name).status["ready"]:
    time.sleep(2)

index = pc.Index(index_name)

In [40]:
from langchain_pinecone import PineconeVectorStore

vector_store = PineconeVectorStore(index=index, embedding=embeddings)

In [41]:
vector_store.add_documents(documents=chunks)

['684ecaa4-36d7-4ed6-a242-beee8ddc6eaf',
 '6dfa5e46-b08d-47d1-8127-0daed3eb7a3b',
 'c002ebae-e831-47bc-b9ff-0e382fd2a26f',
 'fae24bdc-cb7b-4d75-be0c-c3b6f2f645d5',
 '2d9304c1-d523-4797-98b2-092b7d03051a',
 '0579ada3-966b-4c4e-baf8-3aacc062261a',
 '8715205d-1800-4256-bde1-e48089486f1a',
 '6aeba30b-109e-45cc-aedc-30f788914aac',
 '2a56c5c1-dfcc-4547-b76c-7ba8594afd8a',
 '8e5c0c9b-61ad-46e3-b338-e22571159a17']

In [44]:
# Query Directly
results = vector_store.similarity_search(
    "What is artificial intelligence?",
    k=2,
    filter={"source": "AI Introduction"},
)

for res in results:
    print(f"* {res.page_content} [{res.metadata}]")

* Artificial Intelligence (AI) is the simulation of human intelligence in machines.
        These systems are designed to think like humans and mimic their actions.
        AI can be categorized into narrow AI and general AI. [{'page': 1.0, 'source': 'AI Introduction', 'topic': 'AI'}]


In [45]:
# Retriever
retriever = vector_store.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={"k": 1, "score_threshold": 0.4},
)
retriever.invoke("How are AI, machine learning, and deep learning related?", filter={"source": "ML Basics"})

[Document(id='6dfa5e46-b08d-47d1-8127-0daed3eb7a3b', metadata={'page': 1.0, 'source': 'ML Basics', 'topic': 'ML'}, page_content='Machine Learning is a subset of AI that enables systems to learn from data.\n        Instead of being explicitly programmed, ML algorithms find patterns in data.\n        Common types include supervised, unsupervised, and reinforcement learning.')]

# Qdrant Vector Store

In [62]:
!uv pip install -q qdrant-client langchain-qdrant

In [63]:
import os
from uuid import uuid4
from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient, models

In [64]:


COLLECTION_NAME = "company-policy-rag"


# ====================================================================
# 3. CONNECT TO QDRANT (cloud) AND CREATE THE COLLECTION
# ====================================================================
qdrant_api_key = userdata.get("qdrant_api_key")
qdrant_cluster_endpoint = userdata.get("qdrant_cluster_endpoint")

client = QdrantClient(api_key=qdrant_api_key, url=qdrant_cluster_endpoint)

# For a fully local, no-cloud alternative instead of the two lines above:
#   client = QdrantClient(path="./qdrant_data")

if not client.collection_exists(COLLECTION_NAME):
    client.create_collection(
        collection_name=COLLECTION_NAME,
        vectors_config=models.VectorParams(
            size=384,
            distance=models.Distance.COSINE,
        ),
    )

In [65]:

# ====================================================================
# 4. LANGCHAIN VECTOR STORE WRAPPER
# ====================================================================
vector_store = QdrantVectorStore(
    client=client,
    collection_name=COLLECTION_NAME,
    embedding=embeddings,
)


# ====================================================================
# 5. ADD DOCUMENTS
# ====================================================================
chunk_ids = [str(uuid4()) for _ in chunks]  # use stable/deterministic IDs in production

inserted_ids = vector_store.add_documents(documents=chunks, ids=chunk_ids)
print("Inserted chunks:", len(inserted_ids))

Inserted chunks: 10


In [69]:
# ====================================================================
# 6. SIMILARITY SEARCH
# ====================================================================
query = "What are some real-world applications of AI?"
results = vector_store.similarity_search(query, k=4)

for rank, document in enumerate(results, start=1):
    print(f"\nResult {rank}")
    print("Content:", document.page_content[:300])
    print("Page:", document.metadata.get("page_label"))



Result 1
Content: Artificial Intelligence (AI) is the simulation of human intelligence in machines.
        These systems are designed to think like humans and mimic their actions.
        AI can be categorized into narrow AI and general AI.
Page: None

Result 2
Content: Computer Vision is a field of AI that enables computers to understand and analyze images and videos.
        Deep learning models such as convolutional neural networks are commonly used for visual tasks.
        Applications include object detection, facial recognition, medical image analysis, and i
Page: None

Result 3
Content: Machine Learning is a subset of AI that enables systems to learn from data.
        Instead of being explicitly programmed, ML algorithms find patterns in data.
        Common types include supervised, unsupervised, and reinforcement learning.
Page: None

Result 4
Content: Generative AI is a branch of artificial intelligence that can create new content based on learned patterns.
        Genera

In [68]:
# ====================================================================
# 7. RETRIEVER — build a RAG context string
# ====================================================================
retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 4})

context_documents = retriever.invoke("What are the applications of NLP?")
context = "\n\n".join(doc.page_content for doc in context_documents)
print(context)

Natural Language Processing (NLP) is a branch of AI that helps computers understand human language.
        It combines computational linguistics with machine learning and deep learning models.
        Applications include chatbots, translation, sentiment analysis, and text summarization.

Generative AI is a branch of artificial intelligence that can create new content based on learned patterns.
        Generative models can produce text, images, audio, video, and computer code.
        Large language models are a major example of generative AI used for generating and understanding text.
        Popular applications include AI assistants, content generation, code generation, and document summarization.

Deep Learning is a subset of machine learning based on artificial neural networks.
        It uses multiple layers to progressively extract higher-level features from raw input.
        Deep learning has revolutionized computer vision, NLP, and speech recognition.

Machine Learning is a

In [ ]:


# ====================================================================
# NOTE: Qdrant's data model, in short
# ====================================================================
# Collection  -> like a table; holds many Points.
# Point       -> one record = an ID + a vector + a JSON "payload" (metadata).
# Payload     -> arbitrary JSON attached to each point, natively filterable
#               (category, source, page, user_id, etc.) — no external wrapper
#               needed, unlike FAISS.
# Index       -> HNSW graph search under the hood for fast approximate retrieval.
# Storage     -> in-memory, on-disk, or fully managed cloud (what this script uses).

# ASTRA-DB Vector Store

In [51]:
# Run in Google Colab
!uv pip install \
    "langchain>=0.3.23,<0.4" \
    "langchain-core>=0.3.52,<0.4" \
    "langchain-astradb>=0.6,<0.7"

Using Python 3.12.13 environment at: /usr
Resolved 60 packages in 533ms
Prepared 17 packages in 1.14s
Uninstalled 6 packages in 44ms
Installed 17 packages in 178ms
 + astrapy==2.3.1
 + asttokens==3.0.2
 + dnspython==2.8.0
 + executing==2.2.1
 - ipython==7.34.0
 + ipython==9.16.1
 + ipython-pygments-lexers==1.1.1
 + jedi==0.20.0
 - langchain==1.3.13
 + langchain==0.3.30
 + langchain-astradb==0.6.1
 - langchain-core==1.5.4
 + langchain-core==0.3.86
 - langchain-text-splitters==1.1.2
 + langchain-text-splitters==0.3.11
 - psutil==5.9.5
 + psutil==7.2.2
 + pure-eval==0.2.3
 + pymongo==4.17.0
 + stack-data==0.6.3
 - traitlets==5.7.1
 + traitlets==5.16.1
 + uuid6==2025.0.1


In [55]:
# Config
import os
ASTRA_DB_API_ENDPOINT=userdata.get("ASTRA_DB_API_ENDPOINT")
ASTRA_DB_APPLICATION_TOKEN=userdata.get("ASTRA_DB_APPLICATION_TOKEN")

In [58]:
from langchain_astradb import AstraDBVectorStore

vector_store =  AstraDBVectorStore(
    embedding=embeddings,
    api_endpoint=ASTRA_DB_API_ENDPOINT,
    collection_name="astra_vector_langchain",
    token=ASTRA_DB_APPLICATION_TOKEN,
    namespace=None
)

vector_store

In [59]:
vector_store.add_documents(documents=chunks)

['5a7f49613dd84b8b9c605ab14dbc10ac',
 '581a4db835e94118a8b527d49c6029a7',
 '33249d72642f4210bc71e19afe7962af',
 'fa58565374e1495db50518391c49364c',
 'a309f53436734c7dbaf02b0fc1f376bb',
 'd1c8d922e0a847849e2fb2b7cc00a28a',
 '39cc2ce9a15d4dcc8a654dcc76e27406',
 'a1dc4fee7b0a409ba3fb33d0532ebdaf',
 'b0c73ef91d474c07a33d31a31d3c75ad',
 '7754667b2a704b92aed4b21265fef13b']

In [60]:
vector_store.similarity_search("What is the machine learning")

[Document(id='581a4db835e94118a8b527d49c6029a7', metadata={'source': 'ML Basics', 'page': 1, 'topic': 'ML', 'text': 'Machine Learning is a subset of AI that enables systems to learn from data.\n        Instead of being explicitly programmed, ML algorithms find patterns in data.\n        Common types include supervised, unsupervised, and reinforcement learning.'}, page_content='Machine Learning is a subset of AI that enables systems to learn from data.\n        Instead of being explicitly programmed, ML algorithms find patterns in data.\n        Common types include supervised, unsupervised, and reinforcement learning.'),
 Document(id='a309f53436734c7dbaf02b0fc1f376bb', metadata={'source': 'Supervised Learning', 'page': 2, 'topic': 'ML', 'text': 'Supervised learning is a type of machine learning where models learn from labeled data.\n        Each training example contains an input and its corresponding correct output.\n        Common supervised learning tasks include classification and 